# RAGAS 评估 Notebook

评估 RAG Pipeline 的 5 个维度：
- **Faithfulness** — 回答是否基于检索文档（没编造）
- **ContextPrecision** — 检索文档是否相关
- **ContextRecall** — 标准答案信息是否都被检索到
- **AnswerRelevancy** — 回答是否切题
- **FactualCorrectness** — 回答事实是否正确

**前置条件**：
1. `pip install ragas`
2. `.env` 中配置了 `DEEPSEEK_API_KEY`
3. `data/eval_golden_qa.json` 中的 `reference_answer` 已填写

In [6]:
# === 1. 初始化 ===
import sys, json, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from config import DEEPSEEK_API_KEY, LLM_BASE_URL, LLM_MODEL, BASE_DIR, EMBEDDING_MODEL
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import HuggingFaceEmbeddings
from ragas.metrics.collections import (
    Faithfulness, ContextPrecision, ContextRecall,
    AnswerRelevancy, FactualCorrectness,
)

# 加载 Golden QA
with open(Path.cwd() / "data" / "eval_golden_qa.json", "r", encoding="utf-8") as f:
    golden_qa = json.load(f)

# Evaluator LLM + Embeddings
client = AsyncOpenAI(api_key=DEEPSEEK_API_KEY, base_url=LLM_BASE_URL)
evaluator_llm = llm_factory(LLM_MODEL, client=client)
evaluator_embeddings = HuggingFaceEmbeddings(model=EMBEDDING_MODEL)

metrics = {
    "faithfulness": Faithfulness(llm=evaluator_llm),
    "context_precision": ContextPrecision(llm=evaluator_llm),
    "context_recall": ContextRecall(llm=evaluator_llm),
    "answer_relevancy": AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings)
}

# 检查 ChromaDB，为空则自动加载 10-K PDF
from core.vector_store import vector_store
from services.document_service import process_upload

res = vector_store.vectorstore.get(include=["metadatas"])
if not res.get("metadatas"):
    print("ChromaDB 为空，正在加载 10-K PDF...")
    for pdf in sorted((BASE_DIR / "10-k").glob("*.pdf")):
        print(f"  加载: {pdf.name}")
        process_upload(str(pdf), pdf.name)
    print("加载完成！")
else:
    print(f"ChromaDB 已有 {len(res['metadatas'])} 条记录")

print(f"\nGolden QA: {len(golden_qa)} 条 | 指标: {list(metrics.keys())}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8447.66it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ChromaDB 已有 265 条记录

Golden QA: 7 条 | 指标: ['faithfulness', 'context_precision', 'context_recall', 'answer_relevancy']


In [7]:
# === 2. 运行评估 ===
from workflows.rag_pipeline import rag_pipeline

all_results = []

for i, tc in enumerate(golden_qa):
    ref = tc["reference_answer"]
    if ref.startswith("PLACEHOLDER"):
        print(f"[{i+1}/{len(golden_qa)}] SKIP: {tc['question'][:50]}...")
        continue

    # 1. 运行 RAG Pipeline
    state = rag_pipeline.invoke({"question": tc["question"], "retry_count": 0, "workflow_steps": []})
    response = state.get("answer", "")
    contexts = [doc.page_content for doc in state.get("documents", [])]

    # 2. RAGAS 打分
    scores = {}
    kwargs_map = {
        "faithfulness":        {"user_input": tc["question"], "response": response, "retrieved_contexts": contexts},
        "context_precision":   {"user_input": tc["question"], "reference": ref,     "retrieved_contexts": contexts},
        "context_recall":      {"user_input": tc["question"], "retrieved_contexts": contexts, "reference": ref},
        "answer_relevancy":    {"user_input": tc["question"], "response": response}
    }

    for name, scorer in metrics.items():
        try:
            result = await scorer.ascore(**kwargs_map[name])
            scores[name] = result.value
        except Exception as e:
            print(f"  [WARN] {name}: {e}")
            scores[name] = float("nan")

    # 3. 输出：问题 + 回答 + 评分
    print("=" * 80)
    print(f"[{i+1}/{len(golden_qa)}] {tc['id']} ({tc['category']})")
    print(f"Q: {tc['question']}")
    print("-" * 40)
    print(f"A: {response}")
    print("-" * 40)
    score_line = " | ".join(f"{k}={v:.3f}" for k, v in scores.items())
    print(f"评分: {score_line}")
    print()

    all_results.append({"id": tc["id"], "category": tc["category"], "response": response[:200], **scores})
    time.sleep(2)

print("=" * 80)
print(f"评估完成！共 {len(all_results)} 条有效结果")


[QUERY_REWRITER] Original: What was Apple's total net sales in fiscal year 2025?
[QUERY_REWRITER] Rewritten: What was Apple's total net sales, also referred to as total revenue, net revenue, net sales, or total sales from products and services, for the fiscal year ended in 2025, reported in millions or billions of United States dollars (USD)?
[METADATA_EXTRACTOR] companies=['apple'], year=2025, quarter=

[RETRIEVER] Query: What was Apple's total net sales, also referred to as total revenue, net revenue, net sales, or tota
[RETRIEVER] Companies: ['apple']
[RETRIEVER] Filters: {'year': '2025'}
[RETRIEVER] Needs financials: True | Needs risk: False
[RETRIEVER] Hybrid reranked: 10 docs (alpha=0.7)
  Doc 0: company=apple section=item_8_financials page=39 type=text text=The following table shows disaggregated net sales, as well as the portion of tot...
  Doc 1: company=apple section=item_8_financials page=51 type=text text=The following tables show net sales for 2025, 2024 and 2023 and long

In [8]:
# === 3. 结果报告 ===
import numpy as np

if not all_results:
    print("无评价结果。请先填写 eval_golden_qa.json 中的 reference_answer。")
else:
    metric_cols = list(metrics.keys())


    # 汇总平均分
    print(f"\n{'=' * 80}")
    print("汇总分数 (Mean)")
    print("=" * 80)
    for m in metric_cols:
        vals = [r[m] for r in all_results if m in r and not np.isnan(r.get(m, float("nan")))]
        if vals:
            print(f"  {m:25s}: {np.mean(vals):.4f} (n={len(vals)})")
        else:
            print(f"  {m:25s}: N/A")

    # 保存
    out = Path.cwd() / "data" / "eval_results.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    print(f"\n结果已保存: {out}")


汇总分数 (Mean)
  faithfulness             : 0.6991 (n=7)
  context_precision        : 0.7684 (n=7)
  context_recall           : 0.8810 (n=7)
  answer_relevancy         : 0.9920 (n=7)

结果已保存: c:\Users\14869\Desktop\FIN_rag\backend\data\eval_results.json
